In [1]:
#import libraries

import pandas as pd
from collections import Counter
from tqdm import tqdm

In [2]:
#load extracted entities

entity_df=pd.read_csv(
    "../Data/extracted_entities.csv"
)

print(entity_df.shape)

display(entity_df.head())

(9276, 5)


,row_id,original_text,entity_text,entity_label,confidence
0,0,\n Patient Query:\n I woke up this morni...,panadol,medication,0.6307
1,0,\n Patient Query:\n I woke up this morni...,benign paroxysmal positional vertigo,medical_condition,0.7313
2,0,\n Patient Query:\n I woke up this morni...,BPPV,medical_condition,0.5871
3,0,\n Patient Query:\n I woke up this morni...,peripheral vertigo,medical_condition,0.5706
4,0,\n Patient Query:\n I woke up this morni...,dizziness,symptom,0.8374


In [3]:
#check entity labels

print(
    entity_df["entity_label"]
    .value_counts()
)

entity_label
body_part            2440
symptom              1915
medication           1656
medical_condition    1210
disease              1185
treatment             870
Name: count, dtype: int64


In [4]:
#group conversations

grouped_data=entity_df.groupby(
    "row_id"
)

In [5]:
#relationship storage

relationships=[]

In [6]:
#create filtered relationships

for row_id,group in tqdm(grouped_data):

    symptoms=group[
        group["entity_label"]=="symptom"
    ]["entity_text"].str.lower().unique()

    diseases=group[
        group["entity_label"]=="disease"
    ]["entity_text"].str.lower().unique()

    conditions=group[
        group["entity_label"]=="medical_condition"
    ]["entity_text"].str.lower().unique()

    medications=group[
        group["entity_label"]=="medication"
    ]["entity_text"].str.lower().unique()

    #symptom -> disease
    for symptom in symptoms:
        for disease in diseases:
            relationships.append({
                "source":symptom,
                "target":disease,
                "relationship":"RELATED_TO"
            })
    #symptom -> condition
    for symptom in symptoms:
        for condition in conditions:
            relationships.append({
                "source":symptom,
                "target":condition,
                "relationship":"RELATED_TO"
            })
    #condition -> medication
    for condition in conditions:
        for medication in medications:
            relationships.append({
                "source":condition,
                "target":medication,
                "relationship":"TREATED_BY"

            })

100%|███████████████████████████████████████████████████████████████████████████████| 993/993 [00:02<00:00, 402.42it/s]


In [8]:
#create relationships dataframe

relationship_df=pd.DataFrame(
    relationships
)

print(relationship_df.shape)

display(
    relationship_df.head(20)
)

(3650, 3)


,source,target,relationship
0,dizziness,benign paroxysmal positional vertigo,RELATED_TO
1,dizziness,bppv,RELATED_TO
2,dizziness,peripheral vertigo,RELATED_TO
3,giddiness,benign paroxysmal positional vertigo,RELATED_TO
4,giddiness,bppv,RELATED_TO
5,giddiness,peripheral vertigo,RELATED_TO
6,nausea,benign paroxysmal positional vertigo,RELATED_TO
7,nausea,bppv,RELATED_TO
8,nausea,peripheral vertigo,RELATED_TO
9,vomiting,benign paroxysmal positional vertigo,RELATED_TO


In [10]:
#remove duplicate relationships

relationship_df=relationship_df.drop_duplicates()

print(relationship_df.shape)

(3483, 3)


In [11]:
#relationship statistics

print(
    relationship_df["relationship"]
    .value_counts()
)

relationship
RELATED_TO    2260
TREATED_BY    1223
Name: count, dtype: int64


In [12]:
#sample graph relationships

display(
    relationship_df.sample(30)
)

,source,target,relationship
733,cough,lung cancer,RELATED_TO
3406,uti,carbapenems,TREATED_BY
3346,light headed,asthma,RELATED_TO
2717,white line,anemia,RELATED_TO
722,ovulation,progesterone,TREATED_BY
3220,allergies,advil,TREATED_BY
2565,throat infection,painkillers,TREATED_BY
796,flushed,kidney diseases,RELATED_TO
1643,vomiting,diarrhea,RELATED_TO
3103,hair loss,minoxidil solution 2 or 5,TREATED_BY


In [14]:
#save graph relationships

relationship_df.to_csv(
    "../Data/graph_relationships.csv",
    index=False
)

print("graph relationships saved successfully")

graph relationships saved successfully
